# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%%writefile ../scripts/lane_baseline.py
"""My rule baseline (built in w04_baseline_score, ML-07). Frozen once written: w05 onwards import it,
so the baseline is recomputed in the same run as the model and never quietly re-tuned.

The rule in plain words (straight out of the w04 signal audit, momentum was the only signal that held in both client halves):
"A page goes to the top of the review list if it was visible before (100+ impressions in days 31-60), it was already
sliding going into day -30 (days 31-60 more than 20% below days 61-90), and its still under a year old. Pages that tick
all of that are ranked by how much traffic is at stake, biggest first. Pages with zero impressions in days 31-60 go to
the bottom, they cant decline."

Points: sliding = 2, visible = 1, under a year old = 1, so 0-4 points. No fitted weights, nothing learned from labels.
Ties inside a points level are broken by impressions_prev_30d (bigger page first).

Reason codes a page can get (more than one at once):
  already_sliding   days 31-60 impressions more than 20% below days 61-90
  visible_before    100+ impressions in days 31-60
  still_settling    page is under 365 days old (older pages declined less in the audit)
  recent_spike      days 31-60 more than double days 61-90 (could fall back)
  low_volume        1-99 impressions in days 31-60, small numbers swing a lot
  no_prior_traffic  zero impressions in days 31-60, cant be labelled down

Actions: refresh_review (sliding + visible), light_check (sliding but small), watch_spike (recent spike, check
seasonality first), no_action (no prior traffic), monitor (everything else).
"""
import numpy as np
import pandas as pd

RULE_INPUTS = ["impressions_prev_30d", "impr_d61_90", "content_age_days"]


def baseline_points(frame: pd.DataFrame) -> pd.Series:
    sliding = frame["impressions_prev_30d"] < 0.8 * frame["impr_d61_90"]
    visible = frame["impressions_prev_30d"] >= 100
    young = frame["content_age_days"] < 365
    can_decline = frame["impressions_prev_30d"] > 0
    return (can_decline * (2 * sliding + visible + young)).astype(int)


def baseline_score(frame: pd.DataFrame) -> pd.Series:
    # tie-break inside a points level: bigger page first (a percentile, so it never jumps a whole point)
    return baseline_points(frame) + 0.99 * frame["impressions_prev_30d"].rank(pct=True)


def reason_codes(frame: pd.DataFrame) -> pd.Series:
    prev, before = frame["impressions_prev_30d"], frame["impr_d61_90"]
    rules = {
        "already_sliding": prev < 0.8 * before,
        "visible_before": prev >= 100,
        "still_settling": frame["content_age_days"] < 365,
        "recent_spike": prev > 2 * before.clip(lower=1),
        "low_volume": (prev > 0) & (prev < 100),
        "no_prior_traffic": prev == 0,
    }
    codes = pd.Series("", index=frame.index)
    for name, hit in rules.items():
        codes = codes.where(~hit, codes + "|" + name)
    return codes.str.strip("|").replace("", "general_review")


def suggested_action(frame: pd.DataFrame) -> np.ndarray:
    codes = reason_codes(frame)
    return np.select(
        [codes.str.contains("no_prior_traffic"),
         codes.str.contains("already_sliding") & codes.str.contains("visible_before"),
         codes.str.contains("already_sliding"),
         codes.str.contains("recent_spike")],
        ["no_action", "refresh_review", "light_check", "watch_spike"],
        "monitor",
    )

Overwriting ../scripts/lane_baseline.py


In [2]:
"""The rule is in the file above so w05+ can import the exact same frozen baseline. Loading it here plus the features from w03, and checking how many pages land on each points level."""

import sys
sys.path.insert(0, "../scripts")
sys.path.insert(0, "../../scripts")
from pathlib import Path
import numpy as np
import pandas as pd
import lane_features as lf
import lane_baseline as lb

raw = lf.load_raw()
df = lf.build_features(raw)
Y = "is_declining_label"
print(lb.baseline_points(df).value_counts().sort_index().rename("pages").to_string())

0    4101
1    5371
2    7628
3    5548
4    7352


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
"""Score, rank everything, write the CSV. Then evaluate it on the SAME client holdout my model will use later (seed 42, 20% of clients unseen), with the base rate and two floors next to it: a random/dummy ranking, and the reference starter baseline from scripts/02 (recomputed here with its exact formula, I import the helpers from scripts/ml_utils.py and dont edit them).
Once this is written the baseline is frozen, no more tweaking it after the model work starts."""

from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from ml_utils import normalize, percentile_rank

momentum_pct = (df["impressions_prev_30d"] / df["impr_d61_90"].clip(lower=1) - 1) * 100
train, test = lf.client_holdout(df)

queue = pd.DataFrame({
    "content_id": df["content_id"],
    "client_id": df["client_id"],
    "baseline_points": lb.baseline_points(df),
    "reason_codes": lb.reason_codes(df),
    "suggested_action": lb.suggested_action(df),
    "impressions_prev_30d": df["impressions_prev_30d"],
    "impressions_d61_90": df["impr_d61_90"],
    "momentum_pct": momentum_pct.round(1),
    "content_age_days": df["content_age_days"],
    "content_type": df["content_type"],
    "split": np.where(test, "test", "train"),
    Y: df[Y],
})
queue["baseline_score"] = lb.baseline_score(df)
queue["confidence"] = np.select(
    [(queue["baseline_points"] == 4) & (queue["momentum_pct"] <= -25), queue["baseline_points"] >= 3, queue["baseline_points"] >= 1],
    ["high", "medium", "low"], "none",
)
queue = queue.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue.insert(0, "baseline_rank", np.arange(1, len(queue) + 1))

out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)
queue.to_csv(out_dir / "baseline_action_score.csv", index=False)
print(f"wrote {out_dir / 'baseline_action_score.csv'} ({len(queue):,} rows, gitignored like every csv in work/)")
print(queue["suggested_action"].value_counts().to_string())

# evaluation on the held-out clients
reference = (0.40 * percentile_rank(np.log1p(df["impressions_90d"]))
             + 0.30 * percentile_rank(df["days_since_last_update"])
             + 0.25 * (1 - normalize(df["avg_position"].clip(lower=1, upper=50))) * percentile_rank(np.log1p(df["impressions_90d"])) * (df["avg_position"] > 0)
             + 0.05 * (1 - percentile_rank(df["word_count"])) * percentile_rank(np.log1p(df["impressions_90d"])))
mine =queue.set_index("content_id")["baseline_score"].reindex(df["content_id"]).to_numpy()
dummy = DummyClassifier(strategy="stratified", random_state=lf.SEED).fit(np.zeros((train.sum(), 1)), df.loc[train, Y]).predict_proba(np.zeros((test.sum(), 1)))[:, 1]

y = df[Y].to_numpy()
rows = {}
for name, s in [("my rule baseline", mine[test]), ("reference starter baseline (scripts/02)", reference.to_numpy()[test]), ("dummy (random, stratified)", dummy)]:
    rows[name] = {"P@20": lf.precision_at_k(y[test], s, 20), "P@50": lf.precision_at_k(y[test], s, 50),
                  "P@100": lf.precision_at_k(y[test], s, 100), "P@200": lf.precision_at_k(y[test], s, 200),
                  "AUC": roc_auc_score(y[test], s)}
results = pd.DataFrame(rows).T.round(3)
results["test base rate"] = round(y[test].mean(), 3)
print(f"\nheld-out clients: {df.loc[test, 'client_id'].nunique()} | rows {test.sum():,}")
display(results)

wrote ..\outputs\baseline_action_score.csv (30,000 rows, gitignored like every csv in work/)
suggested_action
monitor           9657
refresh_review    9209
light_check       4873
no_action         3388
watch_spike       2873

held-out clients: 6 | rows 2,325


,P@20,P@50,P@100,P@200,AUC,test base rate
my rule baseline,0.70,0.72,0.73,0.670,0.804,0.391
reference starter baseline (scripts/02),0.15,0.24,0.36,0.410,0.627,0.391
"dummy (random, stratified)",0.25,0.32,0.39,0.405,0.505,0.391


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
"""Top 20 of the full queue, one row each: action, reason codes, a confidence note, and what would make the pick wrong. I also show what actually happened in the next 30 days (hindsight, the label) so the review can be checked, the rule itself never saw it.
Client ids are pseudonyms, I swap them for a short index anyway so the table stays readable."""

top20 = queue.head(20).copy()
client_index = {c: f"client_{i:02d}" for i, c in enumerate(queue["client_id"].unique(), start=1)}
top20["client"] = top20["client_id"].map(client_index)
picks_per_client = top20["client"].value_counts()

def what_would_make_it_wrong(row):
    notes = []
    if picks_per_client[row["client"]] >= 5:
        notes.append(f"{picks_per_client[row['client']]} of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same")
    if -25 < row["momentum_pct"] <= -20:
        notes.append("slide is barely past the -20% line, a normal wobble looks like this")
    if row["momentum_pct"] <= -50:
        notes.append("already fell hard before day -30, it may have bottomed out")
    if not notes:
        notes.append("a seasonal dip in keyword demand would look the same, check the keyword trend first")
    return "; ".join(notes)

top20["confidence_note"] = top20.apply(lambda r: f"{r['confidence']} ({r['baseline_points']}/4 points, slide {r['momentum_pct']:.0f}%)", axis=1)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)
top20["declined_next_30d"] = top20[Y].map({1: "yes", 0: "NO"})

with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(top20[["baseline_rank", "client", "impressions_prev_30d", "momentum_pct", "content_age_days", "suggested_action",
                   "reason_codes", "confidence_note", "what_would_make_it_wrong", "declined_next_30d"]])
print(f"top 20 hit rate in hindsight: {top20[Y].mean():.0%} (full-data base rate {df[Y].mean():.0%}) | clients in the top 20: {top20['client'].nunique()}")

,baseline_rank,client,impressions_prev_30d,momentum_pct,content_age_days,suggested_action,reason_codes,confidence_note,what_would_make_it_wrong,declined_next_30d
0,1,client_01,91776,-24.5,224,refresh_review,already_sliding|visible_before|still_settling,"medium (4/4 points, slide -24%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same; slide is barely past the -20% line, a normal wobble looks like this",NO
1,2,client_01,69303,-50.7,225,refresh_review,already_sliding|visible_before|still_settling,"high (4/4 points, slide -51%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same; already fell hard before day -30, it may have bottomed out",yes
2,3,client_01,57172,-26.0,224,refresh_review,already_sliding|visible_before|still_settling,"high (4/4 points, slide -26%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same",yes
3,4,client_02,53136,-22.4,299,refresh_review,already_sliding|visible_before|still_settling,"medium (4/4 points, slide -22%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same; slide is barely past the -20% line, a normal wobble looks like this",yes
4,5,client_03,51661,-21.9,153,refresh_review,already_sliding|visible_before|still_settling,"medium (4/4 points, slide -22%)","slide is barely past the -20% line, a normal wobble looks like this",yes
5,6,client_01,47900,-46.3,224,refresh_review,already_sliding|visible_before|still_settling,"high (4/4 points, slide -46%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same",yes
6,7,client_03,47115,-24.0,313,refresh_review,already_sliding|visible_before|still_settling,"medium (4/4 points, slide -24%)","slide is barely past the -20% line, a normal wobble looks like this",yes
7,8,client_03,46687,-23.3,329,refresh_review,already_sliding|visible_before|still_settling,"medium (4/4 points, slide -23%)","slide is barely past the -20% line, a normal wobble looks like this",NO
8,9,client_04,46296,-25.2,326,refresh_review,already_sliding|visible_before|still_settling,"high (4/4 points, slide -25%)","a seasonal dip in keyword demand would look the same, check the keyword trend first",yes
9,10,client_01,45118,-29.0,224,refresh_review,already_sliding|visible_before|still_settling,"high (4/4 points, slide -29%)","6 of the top 20 are this client, a site-wide event (migration, tracking change, seasonality) would look the same",yes


top 20 hit rate in hindsight: 80% (full-data base rate 54%) | clients in the top 20: 6


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
"""Weak picks first, then the leakage check."""

misses = top20[top20[Y] == 0]
hits = top20[top20[Y] == 1]
print(f"{len(misses)} of the top 20 did NOT decline: ranks {misses['baseline_rank'].tolist()}")
print(f"their slide into day -30: median {misses['momentum_pct'].median():.0f}% vs {hits['momentum_pct'].median():.0f}% for the hits, sizes are about the same "
      f"(median {misses['impressions_prev_30d'].median():,.0f} vs {hits['impressions_prev_30d'].median():,.0f} impressions)")
top50 = queue.head(50)
share = top50["client_id"].value_counts(normalize=True)
print(f"client concentration: top 50 come from {top50['client_id'].nunique()} clients, the biggest two hold {share.iloc[:2].sum():.0%}")
by_client = top20.groupby("client")[Y].agg(picks="size", missed=lambda s: int((s == 0).sum())).sort_values("picks", ascending=False)
display(by_client.T)
by_points = queue.groupby("baseline_points")[Y].agg(declined="mean", pages="size").round(3)
display(by_points)

milder_hits = (hits["momentum_pct"] > misses["momentum_pct"].max()).sum()
print(f"""weak picks, in words:
- I expected the misses to be the mild slides, they arent: {milder_hits} of the hits slid LESS than every miss. on the three things the rule reads (slide, size, age) the misses look just like the hits, so the rule has no way to separate them. thats the ceiling of a 3-input rule, and the gap the model has to close.
- the misses cluster by client: {', '.join(f"{c} missed {r.missed} of {r.picks}" for c, r in by_client[by_client['missed'] > 0].iterrows())}. when every pick from one client misses, thats a client-level thing (site-wide change, seasonality, tracking) not 2 separate page problems. a reviewer should look at the client before the page.
- the top of the list is a few big clients (top 50 from {top50['client_id'].nunique()} clients, two of them hold {share.iloc[:2].sum():.0%}), so the queue is only as good as those clients' data.
- the tie-break ranks by impressions at stake, so a small page thats clearly dying sits far below a big page thats wobbling. thats on purpose (bigger impact) but its a choice, not a fact.
- the points levels do order the risk (declined rate climbs from {by_points.loc[1, 'declined']:.0%} at 1 point to {by_points.loc[4, 'declined']:.0%} at 4), but levels 1-3 are close together, most of the rule's power is 4 points vs 0.""")

# leakage check 1: the score must not move if I scramble every label / label-window / excluded column
label_and_future = ["trend_direction", "trend_pct", "is_declining_label", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                    "days_since_last_update", "freshness_tier", "ctr", "avg_position", "days_with_impressions", "engagement_rate"]
scrambled = df.copy()
rng = np.random.default_rng(lf.SEED)
for col in label_and_future:
    scrambled[col] = rng.permutation(scrambled[col].to_numpy())
same_points = (lb.baseline_points(scrambled) == lb.baseline_points(df)).all()
same_codes = (lb.reason_codes(scrambled) == lb.reason_codes(df)).all()
print(f"\nleakage check 1: scrambled {len(label_and_future)} label/future columns -> points unchanged: {same_points}, reason codes unchanged: {same_codes}")

# leakage check 2: inputs the rule reads, against the excluded list from the w03 contract
rule_inputs = set(lb.RULE_INPUTS)
print("leakage check 2: rule reads", sorted(rule_inputs), "-> overlap with label/future columns:", sorted(rule_inputs & set(label_and_future)))
print("leakage check 3: product flags in the data:", [c for c in raw.columns if any(k in c for k in ["health", "needs_", "quick_win", "priority", "action_type"])], "-> none used")
print("""note on the reference baseline: its score doesnt read the label, but its reason code declining_with_demand is literally trend_direction == "down", which IS the label. so its reason codes and actions describe the answer, not a prediction. mine dont touch trend_direction at all (check 1).
and days 61-90 is backed out as 90d - last_30d - prev_30d, so last_30d is used in that subtraction, but the result is the real days 61-90 count which was known at day -30 (w03 leakage check, test 2 shows the raw 90d is the leaky part, not this).""")

4 of the top 20 did NOT decline: ranks [1, 8, 14, 18]
their slide into day -30: median -26% vs -29% for the hits, sizes are about the same (median 42,740 vs 44,212 impressions)
client concentration: top 50 come from 8 clients, the biggest two hold 56%


client,client_01,client_02,client_03,client_04,client_05,client_06
picks,6,6,3,2,2,1
missed,1,0,1,0,2,0


,declined,pages
baseline_points,,
0,0.083,4101
1,0.557,5371
2,0.574,7628
3,0.587,5548
4,0.720,7352


weak picks, in words:
- I expected the misses to be the mild slides, they arent: 4 of the hits slid LESS than every miss. on the three things the rule reads (slide, size, age) the misses look just like the hits, so the rule has no way to separate them. thats the ceiling of a 3-input rule, and the gap the model has to close.
- the misses cluster by client: client_01 missed 1 of 6, client_03 missed 1 of 3, client_05 missed 2 of 2. when every pick from one client misses, thats a client-level thing (site-wide change, seasonality, tracking) not 2 separate page problems. a reviewer should look at the client before the page.
- the top of the list is a few big clients (top 50 from 8 clients, two of them hold 56%), so the queue is only as good as those clients' data.
- the tie-break ranks by impressions at stake, so a small page thats clearly dying sits far below a big page thats wobbling. thats on purpose (bigger impact) but its a choice, not a fact.
- the points levels do order the risk (decl


leakage check 1: scrambled 12 label/future columns -> points unchanged: True, reason codes unchanged: True
leakage check 2: rule reads ['content_age_days', 'impr_d61_90', 'impressions_prev_30d'] -> overlap with label/future columns: []
leakage check 3: product flags in the data: [] -> none used
note on the reference baseline: its score doesnt read the label, but its reason code declining_with_demand is literally trend_direction == "down", which IS the label. so its reason codes and actions describe the answer, not a prediction. mine dont touch trend_direction at all (check 1).
and days 61-90 is backed out as 90d - last_30d - prev_30d, so last_30d is used in that subtraction, but the result is the real days 61-90 count which was known at day -30 (w03 leakage check, test 2 shows the raw 90d is the leaky part, not this).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.